In [ ]:
!uv sync


In [1]:
def benchmark_basics_transformer(
    do_backward: bool,
    w: int,
    n: int,
    hyperparams: dict[str, int | float] | None = None,
    batch_size: int = 4,
    learning_rate: float = 3e-4,
    device: str | None = None,
) -> dict[str, object]:
    import statistics
    import sys
    import timeit
    from pathlib import Path

    import torch
    import torch.nn.functional as F

    def find_repo_root(start: Path) -> Path:
        for candidate in [start, *start.parents]:
            if (candidate / "basics" / "basics" / "model.py").exists():
                return candidate
        raise FileNotFoundError("Could not find the repo root containing basics/basics/model.py")

    repo_root = find_repo_root(Path.cwd().resolve())
    basics_package_root = repo_root / "basics"
    if str(basics_package_root) not in sys.path:
        sys.path.insert(0, str(basics_package_root))

    from basics.model import BasicsTransformerLM

    hyperparams = hyperparams or {
        "vocab_size": 10_000,
        "context_length": 128,
        "d_model": 512,
        "num_layers": 8,
        "num_heads": 8,
        "d_ff": 2048,
        "rope_theta": 10_000.0,
    }
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

    model = BasicsTransformerLM(**hyperparams).to(torch.device(device))
    raw_batch = torch.randint(
        0,
        hyperparams["vocab_size"],
        (batch_size, hyperparams["context_length"] + 1),
        device=next(model.parameters()).device,
        dtype=torch.long,
    )
    input_ids, labels = raw_batch[:, :-1], raw_batch[:, 1:]
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate) if do_backward else None
    model.train(mode=do_backward)

    def synchronize() -> None:
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    def run_step() -> tuple[float, float]:
        if do_backward:
            optimizer.zero_grad(set_to_none=True)

        synchronize()
        forward_start = timeit.default_timer()
        logits = model(input_ids)
        loss = F.cross_entropy(logits.reshape(-1, hyperparams["vocab_size"]), labels.reshape(-1))
        synchronize()
        forward_time = timeit.default_timer() - forward_start

        backward_time = 0.0
        if do_backward:
            backward_start = timeit.default_timer()
            loss.backward()
            optimizer.step()
            synchronize()
            backward_time = timeit.default_timer() - backward_start

        return forward_time, backward_time

    for _ in range(w):
        run_step()
    timings = [run_step() for _ in range(n)]
    forward_times = [forward_time for forward_time, _ in timings]
    backward_times = [backward_time for _, backward_time in timings]
    step_times = [forward_time + backward_time for forward_time, backward_time in timings]
    forward_mean_ms = statistics.mean(forward_times) * 1_000
    forward_std_ms = statistics.pstdev(forward_times) * 1_000
    backward_mean_ms = statistics.mean(backward_times) * 1_000
    backward_std_ms = statistics.pstdev(backward_times) * 1_000
    mean_ms = statistics.mean(step_times) * 1_000
    std_ms = statistics.pstdev(step_times) * 1_000

    print(f"device: {next(model.parameters()).device}")
    print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"forward mean: {forward_mean_ms:.2f} ms")
    print(f"forward std: {forward_std_ms:.2f} ms")
    print(f"backward mean: {backward_mean_ms:.2f} ms")
    print(f"backward std: {backward_std_ms:.2f} ms")
    print(f"mean step time: {mean_ms:.2f} ms")
    print(f"std step time: {std_ms:.2f} ms")

    model.eval()
    return {
        "model": model,
        "hyperparams": hyperparams,
        "forward_times_s": forward_times,
        "backward_times_s": backward_times,
        "step_times_s": step_times,
        "mean_forward_time_ms": forward_mean_ms,
        "std_forward_time_ms": forward_std_ms,
        "mean_backward_time_ms": backward_mean_ms,
        "std_backward_time_ms": backward_std_ms,
        "mean_step_time_ms": mean_ms,
        "std_step_time_ms": std_ms,
        "do_backward": do_backward,
        "warmup_steps": w,
        "timed_steps": n,
    }


In [24]:
# small

results = benchmark_basics_transformer(
    do_backward=True,
    w=5,
    n=10,
    hyperparams={
        "vocab_size": 10_000,
        "context_length": 128,
        "d_model": 512,
        "num_layers": 8,
        "num_heads": 8,
        "d_ff": 2048,
        "rope_theta": 10_000.0,
    },
)


device: mps:0
parameters: 43,803,136
forward mean: 31.24 ms
forward std: 0.83 ms
backward mean: 66.81 ms
backward std: 1.28 ms
mean step time: 98.04 ms
std step time: 0.85 ms


In [25]:
# medium

results = benchmark_basics_transformer(
    do_backward=True,
    w=5,
    n=10,
    hyperparams={
        "vocab_size": 10_000,
        "context_length": 128,
        "d_model": 768,
        "num_layers": 12,
        "num_heads": 12,
        "d_ff": 3072,
        "rope_theta": 10_000.0,
    },
)


device: mps:0
parameters: 128,625,408
forward mean: 59.29 ms
forward std: 1.21 ms
backward mean: 197.19 ms
backward std: 1.60 ms
mean step time: 256.48 ms
std step time: 2.32 ms


In [26]:
# large

results = benchmark_basics_transformer(
    do_backward=True,
    w=5,
    n=10,
    hyperparams={
        "vocab_size": 10_000,
        "context_length": 128,
        "d_model": 1024,
        "num_layers": 24,
        "num_heads": 16,
        "d_ff": 4096,
        "rope_theta": 10_000.0,
    },
)


KeyboardInterrupt: 

In [2]:
# no warmup steps 

results = benchmark_basics_transformer(
    do_backward=True,
    w=0,
    n=10,
    hyperparams={
        "vocab_size": 10_000,
        "context_length": 128,
        "d_model": 512,
        "num_layers": 8,
        "num_heads": 8,
        "d_ff": 2048,
        "rope_theta": 10_000.0,
    },
)

device: mps:0
parameters: 43,803,136
forward mean: 63.75 ms
forward std: 101.03 ms
backward mean: 72.06 ms
backward std: 25.64 ms
mean step time: 135.81 ms
std step time: 125.55 ms


In [3]:
# few warmup steps

results = benchmark_basics_transformer(
    do_backward=True,
    w=2,
    n=10,
    hyperparams={
        "vocab_size": 10_000,
        "context_length": 128,
        "d_model": 512,
        "num_layers": 8,
        "num_heads": 8,
        "d_ff": 2048,
        "rope_theta": 10_000.0,
    },
)

device: mps:0
parameters: 43,803,136
forward mean: 30.51 ms
forward std: 0.45 ms
backward mean: 66.74 ms
backward std: 1.19 ms
mean step time: 97.25 ms
std step time: 1.45 ms


In [4]:
# 10 warmup steps 

results = benchmark_basics_transformer(
    do_backward=True,
    w=10,
    n=10,
    hyperparams={
        "vocab_size": 10_000,
        "context_length": 128,
        "d_model": 512,
        "num_layers": 8,
        "num_heads": 8,
        "d_ff": 2048,
        "rope_theta": 10_000.0,
    },
)

device: mps:0
parameters: 43,803,136
forward mean: 34.01 ms
forward std: 0.79 ms
backward mean: 74.27 ms
backward std: 1.96 ms
mean step time: 108.28 ms
std step time: 2.65 ms
